# YOLO Full Training & Hyperparameter Optimization Pipeline (Kaggle)

---

## Before You Start

### Enable GPU

1. Open **Notebook Settings**.
2. Set **Accelerator → GPU**.
3. Save changes.

Training on CPU will be significantly slower.

---

## Required Inputs

Add the following Kaggle Datasets as Notebook Inputs:

### Dataset

```
dataset_final_preprocessed
```

Expected structure:

```
dataset_final_preprocessed/
├── train/
├── valid/
├── test/
└── data.yaml
```

### Optuna Database (Optional)

To continue a previous hyperparameter search:

```
acne_hparam_search_v3.2.db
```

The notebook will automatically copy the database from `/kaggle/input` to `/kaggle/working` and resume the study.

---

## How to Run

### 1. Run Setup

Execute all setup cells.

The notebook will:

- prepare the dataset
- initialize Optuna
- create required directories
- verify GPU availability

### 2. Run All Functions

Execute all function definitions.

### 3. Configure `main()`

Set the required flags:

```python
RUN_VALIDATION = True
RUN_HPARAM_SEARCH = True
RUN_PRODUCTION = False
```

Examples:

**Continue Optuna Search**

```python
RUN_HPARAM_SEARCH = True
RUN_PRODUCTION = False
```

**Train Production Model**

```python
RUN_HPARAM_SEARCH = False
RUN_PRODUCTION = True
```

### 4. Run Main

---

## Session Recovery

### Optuna

- resumes automatically from the database
- fixes stale RUNNING trials
- continues from the last completed trial

### Training

If a checkpoint exists:

```
runs/<run_name>/weights/last.pt
```

training resumes automatically.

---

## Saving Results

Before ending the session:

### Export Training Runs

```python
export_runs_zip()
```

Creates:

```
/kaggle/working/runs_export.zip
```

### Export Optuna Database

```python
export_optuna()
```

Creates:

```
/kaggle/working/optuna_latest.db
```

Both files will appear in the **Output** section and can be downloaded.

---

## Notes

- `/kaggle/input` is read-only.
- `/kaggle/working` is writable but temporary.
- Download exported files before closing the session.

# **1. SETUP**

## Clone repo

In [ ]:
import os

REPO_URL = "https://github.com/kenami0981/DermaAI.git"
REPO_NAME = "DermaAI"

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
%cd {REPO_NAME}
!ls

### Install components

In [ ]:
!pip install ultralytics optuna opencv-python pyyaml tqdm

# **2. ALL FUNCTIONS**

## All imports

In [ ]:
import os
import gc
import yaml
import torch
import optuna
import pandas as pd
import zoneinfo
import shutil

from pathlib import Path
from datetime import datetime
from ultralytics import YOLO


### Dataset safe loader

In [4]:
def load_kaggle_dataset():
    """
    FIXES:
    - Windows paths in data.yaml (D:/...)
    - Kaggle /input read-only issues
    - YOLO dataset resolution errors
    """
    src_root = Path("/kaggle/input/datasets/alipamda/dataset-final-preprocessed/dataset_final_preprocessed")

    # WORKING COPY (WRITEABLE)
    dst_root = Path("/kaggle/working/dataset")
    dst_root.mkdir(parents=True, exist_ok=True)

    # copy only once
    if not (dst_root / "data.yaml").exists():
        print("[DATA] Copying dataset to working directory...")
        shutil.copytree(src_root, dst_root, dirs_exist_ok=True)

    yaml_path = dst_root / "data.yaml"

    # LOAD YAML
    with open(yaml_path, "r") as f:
        data = yaml.safe_load(f)

    # FORCE KAGGLE-COMPATIBLE PATHS
    data["train"] = str(dst_root / "train" / "images")
    data["val"]   = str(dst_root / "valid" / "images")
    data["test"]  = str(dst_root / "test" / "images")

    # REMOVE WINDOWS ABSOLUTE PATHS
    data.pop("path", None)

    # SAVE FIXED YAML
    with open(yaml_path, "w") as f:
        yaml.safe_dump(data, f)

    print("[DATA] Kaggle dataset fixed and ready:", yaml_path)

    return yaml_path

### Paths Configuration

In [ ]:
KAGGLE_INPUT = Path("/kaggle/input")
WORKDIR = Path("/kaggle/working")

PROJECT_ROOT = WORKDIR / "DermaAI" / "Models" / "yolo"
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

RUNS_DIR = PROJECT_ROOT / "runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

# Dataset
DATA_YAML = load_kaggle_dataset()
DATA_DIR = DATA_YAML.parent

# Models
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

YOLO_WEIGHTS = MODELS_DIR / "yolo26s.pt"

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"


# OPTUNA DATABASE
OPTUNA_DIR = WORKDIR / "optuna"
OPTUNA_DIR.mkdir(parents=True, exist_ok=True)

OPTUNA_DB = OPTUNA_DIR / "optuna.db"

# Kaggle Input dataset containing previous Optuna DB
INPUT_DB = Path(
    "/kaggle/input/acne-hparam-search-v3-2-db/acne_hparam_search_v3.2.db"
)

# Restore DB only once
if not OPTUNA_DB.exists():

    if INPUT_DB.exists():
        shutil.copy2(INPUT_DB, OPTUNA_DB)
        print(f"[OPTUNA] Restored database from input")
    else:
        print(f"[OPTUNA] No previous database found")
        print(f"[OPTUNA] A new study will be created")

else:
    print(f"[OPTUNA] Existing working database detected")

STORAGE_URL = f"sqlite:///{OPTUNA_DB}"

print(f"DATA_YAML : {DATA_YAML}")
print(f"RUNS_DIR  : {RUNS_DIR}")
print(f"OPTUNA_DB : {OPTUNA_DB}")

# CHANGE TO YOUR PATH !!! 
INPUT_BEST_PARAMS = Path(
    "/kaggle/input/datasets/alipamda/params-acne-hparam-search-v3-2-trial-14-yaml/params_acne_hparam_search_v3.2_trial_14.yaml"
)

In [ ]:
def validate_project_paths():
    """
    Validate project structure before training.
    """

    print("\n=== PROJECT VALIDATION ===\n")

    paths = {
        "PROJECT_ROOT": PROJECT_ROOT,
        "DATA_DIR": DATA_DIR,
        "DATA_YAML": DATA_YAML,
        "MODELS_DIR": MODELS_DIR,
        "RUNS_DIR": RUNS_DIR,
        "OPTUNA_DB": OPTUNA_DB
    }

    valid = True

    for name, path in paths.items():

        exists = path.exists()

        if not exists:
            status = "MISSING"
        elif path.is_dir():
            status = "DIR OK"
        else:
            status = "FILE OK"

        print(f"{name:<15}: {path}")
        print(f"{'':<15}  Status: {status}\n")

        # critical files
        if name in ["DATA_YAML"] and not exists:
            valid = False

    if not valid:
        print("Validation failed.")
        return False

    # YOLO dataset structure

    train_images = DATA_DIR / "train" / "images"
    train_labels = DATA_DIR / "train" / "labels"

    val_images = DATA_DIR / "valid" / "images"
    val_labels = DATA_DIR / "valid" / "labels"

    required = [
        train_images,
        train_labels,
        val_images,
        val_labels
    ]

    missing = [p for p in required if not p.exists()]

    if missing:

        print("Missing dataset folders:")

        for p in missing:
            print(f" - {p}")

        return False

    print("YOLO structure check: OK")

    
    # Model weights

    if not YOLO_WEIGHTS.exists():

        print(f"WARNING: Weights not found:")
        print(YOLO_WEIGHTS)

        print("Training resume will not be available.")

    # Dataset summary

    train_count = len(list(train_images.glob("*.*")))
    val_count = len(list(val_images.glob("*.*")))

    print(f"\nTrain images: {train_count}")
    print(f"Valid images: {val_count}")

    print("\n=== VALIDATION COMPLETE ===\n")

    return True

## **SMOKE TEST Instruction**

**Step 0: Preparation**

Uncomment the last two lines of the code below!

**Step 1: The "Crash"**

Run the code. Let it finish 3 or 4 trials, then **click the STOP button** in Colab (or "Przerwij wykonywanie kodu"). This simulates a session timeout or a manual interruption.

**Step 2: The "Recovery"**
Run the code again. Look at the first line of the output.
*   It will say: `>>> PROGRESS RECOVERED: 4 trials found`.
*   **Result:** This proves the system is OK - it didn't start over - it simply picked up where it left off.



In [7]:
import optuna
import time
from pathlib import Path

def fast_smoke_test():

    print("\n=== OPTUNA RECOVERY TEST ===\n")

    # Writable Kaggle location
    db_path = Path("/kaggle/working/fast_test.db")

    study = optuna.create_study(
        study_name="fast_recovery_test",
        storage=f"sqlite:///{db_path}",
        load_if_exists=True,
        direction="minimize"
    )

    # Fix stale RUNNING trials after crash
    for t in study.trials:

        if t.state == optuna.trial.TrialState.RUNNING:

            print(f"Fixing stale RUNNING trial {t.number}")

            study.tell(
                trial=t.number,
                state=optuna.trial.TrialState.FAIL
            )

    completed = len([
        t for t in study.trials
        if t.state == optuna.trial.TrialState.COMPLETE
    ])

    print(f"Recovered trials: {len(study.trials)}")
    print(f"Completed trials: {completed}")

    def objective(trial):

        x = trial.suggest_float("x", 0, 10)

        print(f"Trial {trial.number} | x={x:.3f}")

        # Simulate training
        time.sleep(5)

        return (x - 5) ** 2

    TARGET_TRIALS = 20

    remaining = max(0, TARGET_TRIALS - completed)

    print(f"Remaining trials: {remaining}")

    if remaining > 0:
        study.optimize(objective, n_trials=remaining)
    else:
        print("Study already complete.")

    print("\nBest trial:")
    print(study.best_trial.number)
    print(study.best_value)
    print(study.best_params)

    print(f"\nDatabase saved at:\n{db_path}")


# Uncomment to test recovery

# if __name__ == "__main__":
#     fast_smoke_test()

## TRAINING

### IMPORTANT! Make sure the T4 GPU runtime is enabled
This script requires a GPU for efficient training. Go to Runtime (Środowisko wykonawcze) --> Change runtime type (Zmień typ środowiska wykonawczego) and select T4 GPU

### EVEN MORE IMPORTANT:

info for Colab Free (T4 GPU)
* **Max Lifetime:** Up to **12 hours** (but it crashed after 4 in my case).
* **Idle Timeout:** **30–90 minutes**. If you stop interacting or close the tab, the session dies quickly.
* **Availability:** Google can reclaim the GPU at any time if demand is high.
* **Storage:** All local files are **deleted** once the session ends. Make sure to save everything on google drive or your device

In [ ]:
TRAINING_RUN_NAME = "acne_train_production_v3.2"
DEVICE = 0 if torch.cuda.is_available() else "cpu"

def merge_best_params(base_params, best_params):
    merged = base_params.copy()
    merged.update(best_params or {})
    return merged


def train_production(best_params=None):

    print("Device:", DEVICE)
    print("GPU available:", torch.cuda.is_available())

    run_dir = RUNS_DIR / TRAINING_RUN_NAME
    ckpt_path = run_dir / "weights" / "last.pt"

    resume_training = ckpt_path.exists()

    production_overrides = {
        "imgsz": 1280,
        "batch": 4,
        "epochs": 100,
        "patience": 12,
        "close_mosaic": 4,
        "erasing": 0.25,
        "hsv_s": 0.0012,
        "hsv_v": 0.0187
    }

    train_params = {

        "data": str(DATA_YAML),

        "project": str(RUNS_DIR),
        "name": TRAINING_RUN_NAME,
        "exist_ok": True,

        "device": DEVICE,

        "workers": min(4, os.cpu_count()),

        "batch": -1 if torch.cuda.is_available() else 8,

        "epochs": 150,
        "patience": 20,

        "imgsz": 640,

        "cos_lr": True,

        "plots": True,
        "verbose": True,

        "save": True,
        "save_period": 1
    }

    # OPTUNA PARAMS

    if best_params:
        train_params = merge_best_params(
            train_params,
            best_params
        )


    # PRODUCTION OVERRIDES

    train_params = merge_best_params(
        train_params,
        production_overrides
    )

    print("\n=== TRAIN CONFIG ===\n")

    for k, v in train_params.items():
        print(f"{k}: {v}")

    print()

    # RESUME
    if resume_training:

        print(f"[RESUME] {ckpt_path}")

        model = YOLO(str(ckpt_path))

        train_params["resume"] = True

    else:

        print("[START] Fresh training")

        model = YOLO(str(YOLO_WEIGHTS))

        train_params["resume"] = False

    # TRAIN
    try:

        results = model.train(**train_params)

    except KeyboardInterrupt:

        print("[INTERRUPTED]")
        raise

    except Exception as e:

        print(f"[ERROR] {e}")
        raise

    finally:

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        gc.collect()

    # SAVE CONFIG
    weights_dir = run_dir / "weights"

    if weights_dir.exists():

        config_path = weights_dir / "training_params.yaml"

        with open(config_path, "w") as f:
            yaml.safe_dump(train_params, f)

    # EXPORT FOR KAGGLE DOWNLOAD

    export_dir = WORKDIR / "exports"
    export_dir.mkdir(exist_ok=True)

    try:

        best_pt = run_dir / "weights" / "best.pt"
        last_pt = run_dir / "weights" / "last.pt"

        if best_pt.exists():
            shutil.copy2(
                best_pt,
                export_dir / f"{TRAINING_RUN_NAME}_best.pt"
            )

        if last_pt.exists():
            shutil.copy2(
                last_pt,
                export_dir / f"{TRAINING_RUN_NAME}_last.pt"
            )

    except Exception as e:
        print(f"[EXPORT WARNING] {e}")

    print("\nTraining complete.")
    print(f"Run directory: {run_dir}")

    return results

## EVALUATION

In [9]:
def get_metrics_from_csv(run_name):

    csv_path = RUNS_DIR / run_name / "results.csv"

    if not csv_path.exists():
        print(f"[ERROR] results.csv not found: {csv_path}")
        return

    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    print("\n=== LAST EPOCH ===\n")
    print(df.tail(1).T)

    map50_col = next(
        (c for c in df.columns if "mAP50" in c and "95" not in c),
        None
    )

    if map50_col is None:
        print("Could not find mAP50 column.")
        return

    best = df.loc[df[map50_col].idxmax()]

    print("\n=== BEST EPOCH ===\n")

    for col in [
        "epoch",
        "metrics/precision(B)",
        "metrics/recall(B)",
        "metrics/mAP50(B)",
        "metrics/mAP50-95(B)"
    ]:
        if col in df.columns:
            print(f"{col}: {best[col]}")

In [10]:
def show_per_class_metrics(run_name):

    model_path = RUNS_DIR / run_name / "weights" / "best.pt"

    if not model_path.exists():
        print("Model not found.")
        return

    model = YOLO(str(model_path))
    metrics = model.val(data=str(DATA_YAML), verbose=False)

    print("\n=== PER-CLASS METRICS ===\n")

    for i, name in model.names.items():
        print(f"{name}")
        print(f"  mAP50     : {metrics.box.ap50[i]:.4f}")
        print(f"  mAP50-95  : {metrics.box.ap[i]:.4f}")
        print(f"  Precision : {metrics.box.p[i]:.4f}")
        print(f"  Recall    : {metrics.box.r[i]:.4f}")
        print()

### **EVALUATION GUIDE**

| Term | Description | Simple Interpretation |
| :--- | :--- | :--- |
| **Precision** | Accuracy of detections | How many detected items were actually correct? |
| **Recall** | Ability to find objects | How many of the total objects did the model find? |
| **mAP50** | Mean Average Precision | Main metric. Success rate at 50% overlap (IoU). |
| **mAP50-95** | Strict mAP | Measures how perfectly the boxes fit the objects. |

### **YOLO Model Benchmarks**

| Metric | Baseline (Poor) | Target (Good) | Perfect |
| :--- | :--- | :--- | :--- |
| **mAP50** | < 0.30 | 0.50 - 0.70 | > 0.85 |
| **mAP50-95** | < 0.15 | 0.30 - 0.45 | > 0.60 |
| **Precision** | < 0.40 | 0.70 - 0.80 | > 0.90 |
| **Recall** | < 0.30 | 0.60 - 0.75 | > 0.85 |


## Hyperparameter Optimization

In [11]:
import torch

DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)
print("GPU available:", torch.cuda.is_available())


Device: 0
GPU available: True


In [ ]:
def run_hyperparameter_search():

    # Use a fixed study name (no timestamp) to ensure Kaggle sessions run OK
    study_name = f"acne_hparam_search_v3.2"

    # backup db after every trial
    def backup_optuna_db():

        try:
            backup_path = RUNS_DIR / "optuna_backup.db"
            shutil.copy2(OPTUNA_DB, backup_path)

        except Exception as e:
            print(f"[DB BACKUP FAILED] {e}")

    def trial_finished_callback(study, trial):
        backup_optuna_db()

    def objective(trial):

        model = None
        results = None

        params = {

            "data": str(DATA_YAML),
            "project": str(RUNS_DIR),
            "name": f"{study_name}_trial_{trial.number}",
            "exist_ok": True,

            "device": DEVICE,
            "workers": 2,
            "batch": -1 if torch.cuda.is_available() else 8,

            "epochs": 50,
            "patience": 10,
            "imgsz": 960,

            # "optimizer": "AdamW",
            # "optimizer": trial.suggest_categorical("optimizer", ["MuSGD", "AdamW"]),
            "optimizer": "MuSGD",
            "end2end": True,

            "lr0": trial.suggest_float("lr0", 1e-3, 4e-3, log=True),

            "weight_decay": trial.suggest_float("weight_decay", 1e-3, 6e-3, log=True),

            # "mosaic": trial.suggest_float("mosaic", 0.1, 0.55),
            # "mixup": trial.suggest_float("mixup", 0.0, 0.08),

            "mosaic": trial.suggest_float("mosaic", 0.2, 0.7),
            "close_mosaic": 3,
            "mixup": trial.suggest_float("mixup", 0.0, 0.15), 
            "erasing": trial.suggest_float("erasing", 0.05, 0.3),

            "translate": trial.suggest_float("translate", 0.02, 0.10),

            "hsv_s": trial.suggest_float("hsv_s", 0.0, 0.05),
            "hsv_v": trial.suggest_float("hsv_v", 0.0, 0.05),

            "scale": trial.suggest_float("scale", 0.1, 0.6),

            "degrees": trial.suggest_float("degrees", 0.0, 15.0),

            "fliplr": 0.5,

            "erasing": trial.suggest_float("erasing", 0.0, 0.2),

            "save": False,
            "verbose": False,
            "plots": False
        }

        try:

            model = YOLO(str(YOLO_WEIGHTS))

            results = model.train(**params)

            metrics = results.results_dict or {}

            return metrics.get("metrics/mAP50(B)", 0.0)

        except Exception as e:

            print(f"[TRIAL FAILED] {e}")

            return 0.0

        finally:
            if model is not None:
                del model

            if results is not None:
                del results

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            gc.collect()

    study = optuna.create_study(
        study_name=study_name,
        storage=STORAGE_URL,
        load_if_exists=True,
        direction="maximize",
        pruner=optuna.pruners.MedianPruner()
    )

    # recover stale Kaggle trials

    for t in study.trials:
        if t.state == optuna.trial.TrialState.RUNNING:
            print(f"Fixing stale RUNNING trial {t.number}")
            study.tell(
                t.number,
                state=optuna.trial.TrialState.FAIL
            )

    print("\n=== TRIAL STATUS ===\n")

    for t in study.trials:
        print(f"Trial {t.number}: {t.state.name}")

    completed = len(
        [t for t in study.trials if t.state.name == "COMPLETE"]
    )

    failed = len(
        [t for t in study.trials if t.state.name == "FAIL"]
    )

    pruned = len(
        [t for t in study.trials if t.state.name == "PRUNED"]
    )

    print(f"\nCompleted: {completed}")
    print(f"Failed: {failed}")
    print(f"Pruned: {pruned}")

    TARGET_TRIALS = 12

    remaining = max(0,TARGET_TRIALS - completed)
    print(f"\nRemaining trials to run: {remaining}")

    if remaining > 0:

        study.optimize(
            objective,
            n_trials=remaining,
            callbacks=[trial_finished_callback]
        )

    else:

        print("Search already complete.")

    best_path = RUNS_DIR / f"{study_name}_best.yaml"

    # download from Output
    best_path_working = WORKDIR / f"{study_name}_best.yaml"

    with open(best_path, "w") as f:
        yaml.dump(study.best_params, f)

    with open(best_path_working, "w") as f:
        yaml.dump(study.best_params, f)

    # save db snapshot after study
    backup_optuna_db()

    print("\nBEST PARAMS:")
    print(study.best_params)

    print("\nBest params saved:")
    print(best_path)
    print(best_path_working)

    return study.best_params

## SAVE RUN (auto zip entire runs folder for Kaggle download)

In [13]:
def export_runs_zip():

    src = Path("/kaggle/working/DermaAI/Models/yolo/runs")
    zip_path = Path("/kaggle/working/runs_export.zip")

    shutil.make_archive(
        base_name=str(zip_path).replace(".zip", ""),
        format="zip",
        root_dir=src
    )

    print("RUNS exported:", zip_path)


def export_optuna_db():

    shutil.copy2(
        "/kaggle/working/optuna/optuna.db",
        "/kaggle/working/optuna_latest.db"
    )

    print("Optuna DB exported: /kaggle/working/optuna_latest.db")

# **3. MAIN**

### MAIN EXECUTION PIPELINE

In [ ]:
def main():
    """
    Main project execution controller.

    """

    # !!! Configure flags here: !!!

    # Validate project structure and required files.
    RUN_VALIDATION = True # Best to keep it True permanently
    # Run Optuna hyperparameter optimization
    RUN_HPARAM_SEARCH = False 
    # Run full production training
    RUN_PRODUCTION = True  
    # Display overall training metrics from results.csv
    SHOW_METRICS = False
    # Display detailed per-class evaluation metrics
    SHOW_PER_CLASS = False
    # Backup selected run to Google Drive
    BACKUP_TO_DRIVE = False
    # Set True if this is the final cell and you want to close the connection to Drive
    UNMOUNT_DRIVE = False


    # Existing run name
    # Required for evaluation / backup
    RUN_NAME = TRAINING_RUN_NAME

    best_params = None
    best_params_path = RUNS_DIR / "acne_hparam_search_v3.2_best.yaml"

    # RESTORE BEST PARAMS FROM KAGGLE INPUT

    if not best_params_path.exists():
        if INPUT_BEST_PARAMS.exists():
            best_params_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(INPUT_BEST_PARAMS, best_params_path)
            print(f"[PARAMS MOD] Successfully copied best params from input to: {best_params_path}")
        else:
            print(f"[PARAMS MOD] WARNING: Source file not found at {INPUT_BEST_PARAMS}")

    if best_params_path.exists():
        with open(best_params_path, "r") as f:
            best_params = yaml.safe_load(f)
            print("[PARAMS MOD] Best parameters loaded successfully.")

    # Validation
    if RUN_VALIDATION:
        if not validate_project_paths():
            print("Validation failed.")
            return

    # Hyperparameter search
    if RUN_HPARAM_SEARCH:
        best_params = run_hyperparameter_search()

        # AUTO-SAVE AFTER HPO (Kaggle crash-safe)
        hpo_save_path = best_params_path 
        with open(hpo_save_path, "w") as f:
            yaml.dump(best_params, f)
        print(f"[HPO SAVE] Best params saved to: {hpo_save_path}")

        # AUTO-BACKUP OPTUNA DB AFTER HPO
        try:
            shutil.copy2(OPTUNA_DB, "/kaggle/working/optuna_latest.db")
            print("[HPO SAVE] Optuna DB backed up")
        except Exception as e:
            print(f"[HPO SAVE ERROR] {e}")

    # Production training
    produced_run_name = None

    if RUN_PRODUCTION:
        results = train_production(best_params)

        if results:
            produced_run_name = results.save_dir.name

            # AUTO-SAVE AFTER TRAINING
            try:
                run_dir = RUNS_DIR / produced_run_name

                shutil.copy2(run_dir / "results.csv", "/kaggle/working/results_last.csv")
                shutil.copy2(run_dir / "weights" / "best.pt", "/kaggle/working/best_last.pt")
                shutil.copy2(run_dir / "weights" / "last.pt", "/kaggle/working/last_last.pt")

                print("[TRAIN SAVE] Training outputs copied to /kaggle/working")

            except Exception as e:
                print(f"[TRAIN SAVE ERROR] {e}")

    # safe fallback (important in Kaggle crash)
    target_run = produced_run_name if produced_run_name is not None else RUN_NAME

    # Metrics
    if SHOW_METRICS:
        get_metrics_from_csv(target_run)

    # Per-class validation
    if SHOW_PER_CLASS:
        show_per_class_metrics(target_run)

    # Backup training (optional external)
    # if BACKUP_TO_DRIVE:
    #     backup_to_drive(
    #         target_run,
    #         unmount=UNMOUNT_DRIVE
    #     )

    # try:
    #     export_training_run() 
    #     print("[EXPORT] runs zipped successfully")
    # except Exception as e:
    #     print(f"[EXPORT ERROR] {e}")


# ENTRYPOINT
if __name__ == "__main__":
    main()

In [ ]:
import shutil
from pathlib import Path

src = Path("/kaggle/working/DermaAI/Models/yolo/runs")
zip_path = Path("/kaggle/working/runs-export.zip")

shutil.make_archive(
    base_name=str(zip_path).replace(".zip", ""),
    format="zip",
    root_dir=src
)